<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  from gensim.models import Word2Vec
  from tensorflow.keras.preprocessing.text import text_to_word_sequence


### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [ ]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

In [ ]:
# Posibles bandas
os.listdir("./songs_dataset/")

In [ ]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

In [ ]:
print("Cantidad de documentos:", df.shape[0])

### 1 - Preprocesamiento

In [ ]:

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [ ]:
# Demos un vistazo
sentence_tokens[:2]

### 2 - Crear los vectores (word2vec)

In [19]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [ ]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [ ]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [ ]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

In [ ]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

### 3 - Entrenar embeddings

In [ ]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

### 4 - Ensayar

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

In [ ]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

In [ ]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

In [ ]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

In [ ]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

### 5 - Visualizar agrupación de vectores

In [ ]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px
def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0, )
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [ ]:


vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [ ]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [ ]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

### **Si se lee raro, todo esto fue dictado en voz alta!**

## Corpus

- Como corpus elegi la Odisea y la Ilíada del proyecto gutenberg.

In [20]:
import pandas as pd
from tensorflow.keras.preprocessing.text import text_to_word_sequence
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  from gensim.models import Word2Vec

# 1. Leer los archivos (asegúrate de que estén en la misma carpeta o ajusta el path)
with open('iliada.txt', 'r', encoding='utf-8') as f:
    iliada_lines = f.readlines()
    
with open('odisea.txt', 'r', encoding='utf-8') as f:
    odisea_lines = f.readlines()

corpus_epico = iliada_lines + odisea_lines


# Agregamos estos stop words a posteriori porque los resultados estaban muy contaminados.
stopwords_es = set([
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas', 'de', 'del', 'al', 'en', 'y', 'a', 'o', 'u', 
    'que', 'con', 'por', 'para', 'como', 'su', 'sus', 'lo', 'si', 'no', 'ni', 'me', 'mi', 'te', 'es', 
    'se', 'le', 'les', 'esta', 'este', 'esto', 'estos', 'estas', 'pero', 'tan', 'muy', 'ha', 'han', 'hay',
    'tu', 'tus', 'yo', 'nos', 'mas', 'más', 'ya', 'cual', 'cuando', 'allí', 'aquí', 'son', 'ser'
])

sentence_tokens_epico = []
for line in corpus_epico:
    if line.strip():
        # Tokenizamos la línea original
        tokens = text_to_word_sequence(line)
        # FILTRO CLAVE: Guardamos solo las palabras que NO estén en nuestra lista de stopwords
        tokens_limpios = [w for w in tokens if w not in stopwords_es]
        
        # Agregamos la lista a nuestro corpus solo si no quedó vacía
        if tokens_limpios:
            sentence_tokens_epico.append(tokens_limpios)

print("Cantidad total de documentos (líneas) en el corpus:", len(sentence_tokens_epico))

Cantidad total de documentos (líneas) en el corpus: 32773


In [21]:
# sentence_tokens_epico[:10]

In [22]:
# word2ve
w2v_model_epico = Word2Vec(min_count=5,    # ignorar palabras con frecuencia menor a 5
                           window=5,       # contexto un poco más amplio
                           vector_size=126,# dimensionalidad
                           negative=20,    
                           workers=12,      
                           sg=1)           # 1: skipgram

# Construir vocabulario
w2v_model_epico.build_vocab(sentence_tokens_epico)

print("Palabras distintas en el vocabulario:", len(w2v_model_epico.wv.index_to_key))

# Entrenar el modelo (reutilizamos la clase callback que ya tenías definida en el notebook)
w2v_model_epico.train(sentence_tokens_epico,
                      total_examples=w2v_model_epico.corpus_count,
                      epochs=50,
                      compute_loss=True,
                      callbacks=[callback()])

Palabras distintas en el vocabulario: 6682
Loss after epoch 0: 312841.5
Loss after epoch 1: 167815.65625
Loss after epoch 2: 200125.90625
Loss after epoch 3: 175191.8125
Loss after epoch 4: 188978.0625
Loss after epoch 5: 168065.4375
Loss after epoch 6: 168602.375
Loss after epoch 7: 161411.5
Loss after epoch 8: 142153.375
Loss after epoch 9: 139467.375
Loss after epoch 10: 215903.75
Loss after epoch 11: 146053.5
Loss after epoch 12: 133481.0
Loss after epoch 13: 137098.5
Loss after epoch 14: 189831.0
Loss after epoch 15: 132624.75
Loss after epoch 16: 131922.0
Loss after epoch 17: 128819.5
Loss after epoch 18: 109139.75
Loss after epoch 19: 126892.0
Loss after epoch 20: 114925.25
Loss after epoch 21: 125348.5
Loss after epoch 22: 127037.0
Loss after epoch 23: 126640.75
Loss after epoch 24: 162089.5
Loss after epoch 25: 109832.5
Loss after epoch 26: 123553.5
Loss after epoch 27: 117311.25
Loss after epoch 28: 94487.0
Loss after epoch 29: 95667.5
Loss after epoch 30: 114429.5
Loss after

(9096477, 11455700)

In [23]:
w2v_model_epico.save("w2v_model_epico.model")
# w2v_model_epico = Word2Vec.load("w2v_model_epico.model")

In [27]:
terminos_interes = ["aquiles", "dioses", "batalla"]

### Cambie dios por dioses y  obtuve mucho mejores resultados ya que en la mitología griega se suele referir a ellos en plural.

In [28]:
for termino in terminos_interes:
    print(f"--- Análisis de '{termino}' ---")
    
    # MÁS SIMILARES
    print(f"Palabras más similares a '{termino}':")
    similares = w2v_model_epico.wv.most_similar(positive=[termino], topn=5)
    for palabra, similitud in similares:
        print(f"  {palabra}: {similitud:.4f}")
        
    print("\n")
    
    # MENOS SIMILARES
    print(f"Palabras menos similares a '{termino}':")
    menos_similares = w2v_model_epico.wv.most_similar(negative=[termino], topn=5)
    for palabra, similitud in menos_similares:
        print(f"  {palabra}: {similitud:.4f}")
        
    print("\n" + "="*30 + "\n")

--- Análisis de 'aquiles' ---
Palabras más similares a 'aquiles':
  ¡ea: 0.4438
  agenor: 0.4437
  eácida: 0.4435
  despojado: 0.4394
  peonio: 0.4383


Palabras menos similares a 'aquiles':
  imperan: 0.0807
  llegues: 0.0433
  ver: 0.0337
  ventura: 0.0294
  establos: 0.0243


--- Análisis de 'dioses' ---
Palabras más similares a 'dioses':
  bienaventurados: 0.5978
  númenes: 0.5432
  inmortales: 0.5140
  sempiternos: 0.5072
  ¡hija: 0.4815


Palabras menos similares a 'dioses':
  arenga: 0.0789
  polvo: 0.0627
  dientes: 0.0583
  acudió: 0.0487
  tomé: 0.0461


--- Análisis de 'batalla' ---
Palabras más similares a 'batalla':
  propongo: 0.4545
  combatían: 0.4526
  dura: 0.4330
  adelantó: 0.4305
  podemos: 0.4221


Palabras menos similares a 'batalla':
  circe: 0.0839
  xviii: 0.0750
  agradable: 0.0670
  242: 0.0631
  tocó: 0.0526




In [ ]:
# word2ve
w2v_model_epico = Word2Vec(min_count=5,    # ignorar palabras con frecuencia menor a 5
                           window=5,       # contexto un poco más amplio
                           vector_size=126,# dimensionalidad
                           negative=20,    
                           workers=12,      
                           sg=1)           # 1: skipgram

# Construir vocabulario
w2v_model_epico.build_vocab(sentence_tokens_epico)

print("Palabras distintas en el vocabulario:", len(w2v_model_epico.wv.index_to_key))

# Entrenar el modelo (reutilizamos la clase callback que ya tenías definida en el notebook)
w2v_model_epico.train(sentence_tokens_epico,
                      total_examples=w2v_model_epico.corpus_count,
                      epochs=50,
                      compute_loss=True,
                      callbacks=[callback()])

Palabras distintas en el vocabulario: 6682
Loss after epoch 0: 312841.5
Loss after epoch 1: 167815.65625
Loss after epoch 2: 200125.90625
Loss after epoch 3: 175191.8125
Loss after epoch 4: 188978.0625
Loss after epoch 5: 168065.4375
Loss after epoch 6: 168602.375
Loss after epoch 7: 161411.5
Loss after epoch 8: 142153.375
Loss after epoch 9: 139467.375
Loss after epoch 10: 215903.75
Loss after epoch 11: 146053.5
Loss after epoch 12: 133481.0
Loss after epoch 13: 137098.5
Loss after epoch 14: 189831.0
Loss after epoch 15: 132624.75
Loss after epoch 16: 131922.0
Loss after epoch 17: 128819.5
Loss after epoch 18: 109139.75
Loss after epoch 19: 126892.0
Loss after epoch 20: 114925.25
Loss after epoch 21: 125348.5
Loss after epoch 22: 127037.0
Loss after epoch 23: 126640.75
Loss after epoch 24: 162089.5
Loss after epoch 25: 109832.5
Loss after epoch 26: 123553.5
Loss after epoch 27: 117311.25
Loss after epoch 28: 94487.0
Loss after epoch 29: 95667.5
Loss after epoch 30: 114429.5
Loss after

(9096477, 11455700)

Al principio probamos dimensiones de embedding y ventanas más grandes, pero daba pésimo. Puede ser por la dimensionality curse para un vocabulario de 6k palabras.
Luego de jugar un poco con los parametros, llegamos a resultados razonables. Se asocia Patroclo con Aquiles (el mejor amigo o primo creo); dioses con inmortales, bienaventurados; y "batalla" que es la palabra más común muestra 5 palabras muy intuitivas para asociar.

## Reducción dimensional

In [29]:
vecs_epico, labels_epico = reduce_dimensions(w2v_model_epico, num_dimensions=2)


In [ ]:
# word2ve
w2v_model_epico = Word2Vec(min_count=5,    # ignorar palabras con frecuencia menor a 5
                           window=5,       # contexto un poco más amplio
                           vector_size=126,# dimensionalidad
                           negative=20,    
                           workers=12,      
                           sg=1)           # 1: skipgram

# Construir vocabulario
w2v_model_epico.build_vocab(sentence_tokens_epico)

print("Palabras distintas en el vocabulario:", len(w2v_model_epico.wv.index_to_key))

# Entrenar el modelo (reutilizamos la clase callback que ya tenías definida en el notebook)
w2v_model_epico.train(sentence_tokens_epico,
                      total_examples=w2v_model_epico.corpus_count,
                      epochs=50,
                      compute_loss=True,
                      callbacks=[callback()])

Palabras distintas en el vocabulario: 6682
Loss after epoch 0: 312841.5
Loss after epoch 1: 167815.65625
Loss after epoch 2: 200125.90625
Loss after epoch 3: 175191.8125
Loss after epoch 4: 188978.0625
Loss after epoch 5: 168065.4375
Loss after epoch 6: 168602.375
Loss after epoch 7: 161411.5
Loss after epoch 8: 142153.375
Loss after epoch 9: 139467.375
Loss after epoch 10: 215903.75
Loss after epoch 11: 146053.5
Loss after epoch 12: 133481.0
Loss after epoch 13: 137098.5
Loss after epoch 14: 189831.0
Loss after epoch 15: 132624.75
Loss after epoch 16: 131922.0
Loss after epoch 17: 128819.5
Loss after epoch 18: 109139.75
Loss after epoch 19: 126892.0
Loss after epoch 20: 114925.25
Loss after epoch 21: 125348.5
Loss after epoch 22: 127037.0
Loss after epoch 23: 126640.75
Loss after epoch 24: 162089.5
Loss after epoch 25: 109832.5
Loss after epoch 26: 123553.5
Loss after epoch 27: 117311.25
Loss after epoch 28: 94487.0
Loss after epoch 29: 95667.5
Loss after epoch 30: 114429.5
Loss after

(9096477, 11455700)

In [31]:
import pandas as pd
import plotly.express as px
import numpy as np

# Términos obligatorios que queremos ver sí o sí
terminos_interes = ["aquiles", "dioses", "batalla"]
MAX_WORDS = 50

# 1. Asegurar que las palabras de interés entren en el gráfico
indices_a_graficar = list(range(min(MAX_WORDS, len(labels_epico))))
lista_labels_completa = list(labels_epico)

for termino in terminos_interes:
    if termino in lista_labels_completa:
        idx = lista_labels_completa.index(termino)
        if idx not in indices_a_graficar:
            indices_a_graficar.append(idx)

# 2. Armar el DataFrame con los índices asegurados
df_plot = pd.DataFrame({
    'x': vecs_epico[indices_a_graficar, 0],
    'y': vecs_epico[indices_a_graficar, 1],
    'Palabra': labels_epico[indices_a_graficar]
})

df_plot['Grupo'] = df_plot['Palabra'].apply(lambda x: 'Término de Interés' if x in terminos_interes else 'Otras palabras')
df_plot['Tamaño_Punto'] = df_plot['Grupo'].apply(lambda x: 26 if x == 'Término de Interés' else 6)

# TRUCO: Si es palabra de interés, ocultamos su texto por defecto en el scatter 
# para que no se superponga con el cartel flotante (badge) que crearemos después.
df_plot['Palabra_Bold'] = df_plot.apply(
    lambda r: "" if r['Grupo'] == 'Término de Interés' else f"<b>{r['Palabra']}</b>", axis=1
)

# 3. Calcular márgenes dinámicos amplios (15%) para distanciar todo de los bordes
x_margin = (df_plot['x'].max() - df_plot['x'].min()) * 0.15
y_margin = (df_plot['y'].max() - df_plot['y'].min()) * 0.15

# 4. Graficar estirando mucho el ancho (1600px) para maximizar la distancia entre puntos
fig = px.scatter(df_plot, 
                 x='x', 
                 y='y', 
                 text='Palabra_Bold', 
                 color='Grupo',
                 size='Tamaño_Punto',
                 size_max=26,
                 color_discrete_map={'Término de Interés': '#E74C3C', 'Otras palabras': '#BDC3C7'},
                 title="Mapa de Embeddings Optimizado - Ilíada y Odisea")

# Ajuste general del texto de fondo
fig.update_traces(textposition='top center')

# Forzamos el layout ultra ancho y los rangos de los ejes con los colchones calculados
fig.update_layout(
    width=1600, 
    height=850, 
    showlegend=True,
    font=dict(size=12),
    xaxis=dict(range=[df_plot['x'].min() - x_margin, df_plot['x'].max() + x_margin]),
    yaxis=dict(range=[df_plot['y'].min() - y_margin, df_plot['y'].max() + y_margin])
)

# 5. Agregar las etiquetas flotantes estilizadas (Annotations) SIN el parámetro cliponaxis
for idx, row in df_plot[df_plot['Grupo'] == 'Término de Interés'].iterrows():
    fig.add_annotation(
        x=row['x'],
        y=row['y'],
        text=f"<b>{row['Palabra'].upper()}</b>", # Letra en mayúscula y negrita
        showarrow=True,
        arrowhead=2,
        arrowcolor="#E74C3C",
        arrowsize=1.2,
        ax=0,
        ay=-50,               # Distancia hacia arriba para que no pise al punto gigante
        bgcolor="#E74C3C",    # Fondo rojo del cartel
        font=dict(color="white", size=14, family="Arial"), # Letras blancas
        bordercolor="#C0392B", # Borde del cartel
        borderwidth=2,
        borderpad=7           # Padding para darle cuerpo al badge de fondo
    )

fig.show(renderer="colab")

## Conclusiones !

## Sacar las Stop Words en español mejoró un montón la calidad del modelo, permitiendo que el gráfico a lo último refleje con mucho mejor precisión la estructura de las obras. En el último gráfico se observan agrupaciones mucho más coherentes con el texto. Aquiles está muy cerca de Héctor, batalla junto a Menelao. Las duplas padre-hijo y rey-ciudad están prácticamente pegadas, y naves muy cercanas a Mar también palacio pretendientes y muerte estructuran de forma implícita el desenlace de la odisea. Y Telemaco y Ulises también están muy cerca, ya que son padres. Ahí también hubo mucha diferencia al reemplazar Dios por Dioses en los términos de interés. Ya que cuando se usó Dios no se obtuvieron resultados muy interpretables, en cambio cuando se pasó a usar Dioses aparecieron palabras mucho más corrientes. Esto seguramente se debe reflejar en la pluralidad que tienen los dioses en la metodología griega. Generalmente se refiere a ellos por sus nombres o en plural como los dioses del oriente.

## No se refleja en el output del notebook, pero en una primera iteración no había sacado los stop words en español y en los términos de interés estaba Dios en vez de Dioses y los resultados no estaban buenos. No tenían mucha interpretabilidad y el gráfico en particular era muy denso en el centro lo cual quiere decir cuando estamos en un contexto de reducción dimensional es que la varianza apunta para todos lados po0 igual ->  no captura buenas relaciones en baja dimensión. Ahora en este último gráfico están mucho más separadas y se ven clusteres de palabras relacionadas.

**Si se lee raro, todo esto fue dictado en voz alta!**